# Calibration of OZIRIIS: a vector-Zernike WFS, and its deformable mirror.

In this tutorial I will show you how to calibrate OZIRIIS. This instrument has a vector-Zernike WFS, which is two regular Zernike WFSs with different core-phase-shifts. Its deformable mirror is an ALPAO 97.

The calibration procedure is as follows:
1. We load the data, and pre-process it to facilitate the calibration.
2. We make a first rough calibration to place the pupils in the correct positions
3. If we had a good reference frame, we could calibrate the static amplitude and phase from the bench
4. We define the deformable mirror and perform a rough alignment to determine rotations, flips and signs
5. Using the fully differentiable WFS and DM models we fit all the degrees of freedom to compute the misregistration

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

from AI4AO import ZernikeWFS, DeformableMirror, imshow, TwinCalibrator

from mmengine import Config
import numpy as np
import matplotlib.pyplot as plt

You can download the data to calibrate from here

[Link to files](https://nuage.osupytheas.fr/s/Fjo3X6wSZz9XyoM)

We first load the interaction matrix taken in the bench. Given that its shape is not square, we had to pad it.

In [ ]:
imat_data = np.load(r"../../Data/Oziriis/IM_fullframe.npy")
imat_data = torch.from_numpy(imat_data).to(device = device, dtype = torch.float32)

C,W,H = imat_data.shape
pad = int(W - H) // 2
imat_data = torch.nn.functional.pad(imat_data, (pad, pad, 0, 0))

Then, we can load the modes to commands matrix `M2C`.

In [ ]:
M2C = np.load(r"../../Data/Oziriis/M2C_KL.npy")
M2C = torch.from_numpy(M2C).to(device = device, dtype = torch.float32)

If we had a good reference frame we would load it here, but in this case we can just use the interaction matrix to have a good reference for the pupil positions. The static amplitude and phase from the bench will then have to be calibrated using the interaction matrix

In [ ]:
binnary_reference = imat_data.std(dim = 0)
binnary_reference = (binnary_reference > binnary_reference.max() * 0.056).to(device=device, dtype = torch.float32)
binnary_reference /= binnary_reference.sum()

In [ ]:
paramfile = 'Oziriis_params.py'

WFSParams = Config.fromfile(paramfile)['WFSParams']
DMParams = Config.fromfile(paramfile)['DMParams']

In [ ]:
WFSParams["useNoise"] = False
WFSParams["centralObstruction"] = 0

wfs = ZernikeWFS(WFSParams, device)

calibrator = TwinCalibrator(wfs, dm=None, device=device)

## Pupil positioning

We start the calibration by placing the pupils in the correct positions. To do so, we 
1. Update the position of the pupils with `wfs.BuildMask()`
2. Compute a reference intensity with `wfs.BuildReferenceIntensity()`
3. Compare it to the binnary mask with `l = loss(binnary_reference * 1e5,digital_image * 1e5)` (The factor 1e5 is because the loss is too small)
4. Backpropagate with `l. backward() and optimizer.step()`

It is possible that the pupils are not alighend on the first try and you can run the cell as many times as you need. Also, you can change `TrainRunNb` to have more or less iterations, and the learning rate of the optimizer `lr = 5e-3`

In [ ]:
final_loss = calibrator.fit_pupil_to_reference(
    binnary_reference,
    [wfs.positions],
    lr=5e-3,
    n_iter=300,
    loss_fn=lambda ref, dig: torch.nn.MSELoss()(ref * 1e5, dig * 1e5),
)

We can add some prior information, like the pupil depths

In [ ]:
with torch.no_grad():
    wfs.depths[:] = torch.tensor(
        [torch.pi * -0.33, torch.pi * 0.76],
        device=wfs.depths.device
    )
    
wfs.BuildMask()

## DM parameters

`moffatParam`: Corresponds to the moffat exponent which interpolates between a Cauchy and Gaussian shapes for the influence functions.

`signedAmplitude`: Amplitude in OPD of the DM. Can be positive or negative, depending on convetion

`Flips`: Flip left-right or top-bottom. If you don't know them, don't worry as there is a rough calibration step that can find the best configuration

`misreg`: Dictionary containing the misregistrations of the DM. These are compatible with OOPAO's misreg. 

In [ ]:
DMParams = {"Nactuator": 11,
            "moffatParam":2,
            "signedAmplitude": -8e-6,
            "MechCoupling": 0.36, 
            "FlipLeftRight": True,
            "FlipTopBottom": True}

misreg = {}
misreg['rotationAngle'] = 90
# shift X in m
misreg['shiftX'] = 0
# shift Y in m
misreg['shiftY'] = 0
# amamorphosis angle in degrees
misreg['anamorphosisAngle'] = 0
# normal scaling in % of diameter
misreg['tangentialScaling'] = 0
# radial scaling in % of diameter
misreg['radialScaling'] = 0

dm = DeformableMirror(WFSParams,
                      DMParams,
                      device=device,
                      offset_to_fit_number_of_actuators=0.1,
                      misreg=misreg)

calibrator.dm = dm

If you don't know the correct flips, rotations, and sign of the DM, you can use the `dm.RoughCalibration(wfs, imat_data, M2C)` for an automatic selection of the best starting values.

In [ ]:
calibrator.rough_calibrate_dm(imat_data, M2C)

We define the static amplitude and phase maps for the bench. These will get optimized along with the misreg of the DM. To avoid overfitting or falling into local minima, the training is set such that these maps are not updated on the begining and slowly start getting more and more importance as the optimization progresses.

In [ ]:
ref_pupil, ref_phase = calibrator.init_static_offsets()

Always check on some modes to see if the rough aligment makes sense! You can change `idx` to see through some modes

In [ ]:
index = list(range(0,80,1))

calibrator.sanity_check_plot(imat_data, M2C, index, idx=4)

## Optimization of misregistration, WFS parameters and static amplitude and phase

We define the optimization problem to be: find the best parameters for the DM, WFS and offsets to produce a synthetic interaction matrix as close as possible to the one mesured on the bench.

Given that the main parameters to be trained are the DM's, the learning rate is set to 0.01. For the WFS, 0.001, given that it was previously optimized. For the static amplitude and phase, to avoid overfitting, we start the optimization with an extremely low learning rate for them, for the optimizer to priorize changing the parameters of the DM and WFS. The learning rate for the offsets is then gradually increased to match that of the DM and WFS.

In [ ]:
final_loss, original_positons, transformed_positons = calibrator.fit_dm_and_offsets(
    imat_data,
    M2C,
    index,
    n_iter=200,
    lr_dm=1e-2,
    lr_wfs=1e-2,
    lr_offset_start=-10,
    lr_offset_end=-2,
    fit_static_offsets=True,
    plot_mode_idx=4,
)

You can check the retrieved actuator positions, the static amplitude and phase values

In [ ]:
calibrator.plot_actuator_and_offsets(original_positons, transformed_positons)

We now compute a new interaction matrix with the fitter parameters

In [ ]:
modes = calibrator.rebuild_reconstruction_matrix(M2C)

You can change the index to see how well the algorithm worked to fit the parameters

In [ ]:
idx = 1
target_idx = imat_data[idx].cpu().detach().numpy()
calibrator.plot_fit_residual(imat_data, idx, vmin=target_idx.min(), vmax=target_idx.max())

In [ ]:
imshow(torch.stack((imat_data, wfs.iMat, imat_data - wfs.iMat)), max_channel_number=16, same_scale=True)
plt.show()

In [ ]:
cov = calibrator.crosstalk_diagnostic(imat_data)

In [ ]:
PATH_WFS, PATH_DM = calibrator.save("Oziriis", data_dir="../../Data")

In [ ]:
wfs = ZernikeWFS(WFSParams, device)
dm = DeformableMirror(WFSParams,
                    DMParams,
                    device=device)

calibrator.wfs = wfs
calibrator.dm = dm
calibrator.load("Oziriis", data_dir="../../Data")

modes = calibrator.rebuild_reconstruction_matrix(M2C)
cov = calibrator.crosstalk_diagnostic(imat_data)